# OpenAlex DOI Cross-check & Metrics Updater

`OpenAlex_Input.xlsx`의 기존 데이터를 보존한 채 DOI를 기준으로 OpenAlex에서 서지정보, 피인용 수(`Cited by`), FWCI를 조회하여 `OpenAlex_Output.xlsx`를 생성합니다.

- 입력 시트: `DOI 찾기`
- DOI 열: `DOI`
- 기존 A:I는 보존
- OpenAlex 결과는 J열 이후에 추가
- DOI가 없는 행은 조회하지 않음
- DOI는 최대 100건씩 배치 조회


## 1. 패키지 설치
최초 1회만 실행하면 됩니다. 이미 설치되어 있으면 건너뛰어도 됩니다.


In [22]:
%pip install requests openpyxl truststore

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. 라이브러리 및 사용자 설정


In [23]:
import os
import re
import time
from pathlib import Path
from difflib import SequenceMatcher

import truststore
truststore.inject_into_ssl()

import requests
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

print("SSL backend: Windows system certificate store enabled")

# ===== 사용자 설정 =====
INPUT_FILE = 'OpenAlex_Input.xlsx'
OUTPUT_FILE = 'OpenAlex_Output.xlsx'
SHEET_NAME = 'DOI 찾기'
HEADER_ROW = 1

OPENALEX_API_URL = 'https://api.openalex.org/works'
API_KEY = os.getenv('OPENALEX_API_KEY', '').strip()  # 선택 사항

BATCH_SIZE = 100
TIMEOUT = 30
MAX_RETRIES = 4
REQUEST_DELAY = 0.25

# Cross-check 판정 기준
TITLE_OK_THRESHOLD = 0.90
TITLE_REVIEW_THRESHOLD = 0.75
JOURNAL_OK_THRESHOLD = 0.85
JOURNAL_REVIEW_THRESHOLD = 0.65

# 입력 열 이름
H_TITLE = '논문명'
H_AUTHOR = '주저자명'
H_JOURNAL = '학술지명'
H_DATE = '게재일'
H_DOI = 'DOI'

OUTPUT_HEADERS = [
    'OA 논문명',
    'OA 주저자',
    'OA 학술지명',
    'OA 게재일',
    'Cited by',
    'FWCI',
    'OpenAlex ID',
    'OA DOI',
    'OpenAlex Updated',
    'Cross-check',
]

print('API Key 사용:', bool(API_KEY))
print('입력 파일:', Path(INPUT_FILE).resolve())


SSL backend: Windows system certificate store enabled
API Key 사용: True
입력 파일: D:\VSC\OpenAlex\OpenAlex_Input.xlsx


## 3. DOI·문자열 정규화 및 OpenAlex 필드 추출 함수


In [24]:
def normalize_doi(value):
    if value is None:
        return None
    doi = str(value).strip()
    if not doi:
        return None
    doi = re.sub(r'^doi:\s*', '', doi, flags=re.IGNORECASE)
    doi = re.sub(r'^https?://(?:dx\.)?doi\.org/', '', doi, flags=re.IGNORECASE)
    doi = doi.strip().strip('<>[]{}()')
    doi = doi.rstrip('.,;:')
    if not doi.lower().startswith('10.') or '/' not in doi:
        return None
    return doi.lower()


def normalize_text(value):
    if value is None:
        return ''
    text = str(value).lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'[^a-z0-9가-힣]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def compact_person_name(value):
    return re.sub(r'[^a-z0-9가-힣]', '', normalize_text(value))


def similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return None
    return SequenceMatcher(None, a, b).ratio()


def get_year(value):
    if value is None:
        return None
    if hasattr(value, 'year'):
        return value.year
    match = re.search(r'\b(19|20)\d{2}\b', str(value))
    return int(match.group(0)) if match else None


def get_first_author(work):
    authorships = work.get('authorships') or []
    if not authorships:
        return ''
    return ((authorships[0].get('author') or {}).get('display_name') or '')


def get_journal(work):
    primary_location = work.get('primary_location') or {}
    source = primary_location.get('source') or {}
    return source.get('display_name') or ''


## 4. Cross-check 함수
기존 제목·주저자·학술지·게재연도와 OpenAlex 반환값을 비교하여 `OK` 또는 `CHECK: ...`를 생성합니다.


In [25]:
def author_matches(local_authors, oa_first_author):
    if not local_authors or not oa_first_author:
        return None

    local_compact = compact_person_name(local_authors)
    oa_compact = compact_person_name(oa_first_author)

    if oa_compact and oa_compact in local_compact:
        return True

    oa_tokens = [t for t in normalize_text(oa_first_author).split() if len(t) > 1]
    if not oa_tokens:
        return None

    local_text = normalize_text(local_authors)
    hit_count = sum(1 for token in oa_tokens if token in local_text)
    return hit_count == len(oa_tokens)


def make_cross_check(local_title, local_authors, local_journal, local_date,
                     oa_title, oa_first_author, oa_journal, oa_date):
    flags = []

    title_score = similarity(local_title, oa_title)
    if title_score is not None:
        if title_score < TITLE_REVIEW_THRESHOLD:
            flags.append(f'TITLE {title_score:.0%}')
        elif title_score < TITLE_OK_THRESHOLD:
            flags.append(f'TITLE? {title_score:.0%}')

    author_ok = author_matches(local_authors, oa_first_author)
    if author_ok is False:
        flags.append('AUTHOR')

    journal_score = similarity(local_journal, oa_journal)
    if journal_score is not None:
        if journal_score < JOURNAL_REVIEW_THRESHOLD:
            flags.append(f'JOURNAL {journal_score:.0%}')
        elif journal_score < JOURNAL_OK_THRESHOLD:
            flags.append(f'JOURNAL? {journal_score:.0%}')

    local_year = get_year(local_date)
    oa_year = get_year(oa_date)
    if local_year and oa_year and local_year != oa_year:
        flags.append(f'YEAR {local_year}/{oa_year}')

    return 'OK' if not flags else 'CHECK: ' + ' / '.join(flags)


## 5. Excel 처리 함수


In [26]:
def header_map(ws):
    result = {}
    for cell in ws[HEADER_ROW]:
        if cell.value is not None and str(cell.value).strip():
            result[str(cell.value).strip()] = cell.column
    return result


def ensure_output_headers(ws, headers):
    hmap = header_map(ws)
    next_col = ws.max_column + 1
    for name in headers:
        if name not in hmap:
            ws.cell(HEADER_ROW, next_col).value = name
            hmap[name] = next_col
            next_col += 1
    return hmap


def copy_header_style(ws, source_col, target_cols):
    source = ws.cell(HEADER_ROW, source_col)
    for col in target_cols:
        target = ws.cell(HEADER_ROW, col)
        if source.has_style:
            target._style = source._style
        else:
            target.font = Font(bold=True)
            target.fill = PatternFill('solid', fgColor='D9EAF7')


## 6. OpenAlex API 조회 함수


In [27]:
def query_batch(session, dois):
    if not dois:
        return {}

    doi_filter = '|'.join(f'https://doi.org/{doi}' for doi in dois)

    params = {
        'filter': f'doi:{doi_filter}',
        'per_page': 100,
        'select': (
            'id,doi,display_name,publication_date,authorships,'
            'primary_location,cited_by_count,fwci,updated_date'
        ),
    }

    if API_KEY:
        params['api_key'] = API_KEY

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(OPENALEX_API_URL, params=params, timeout=TIMEOUT)

            if response.status_code in {429, 500, 502, 503, 504}:
                wait = min(2 ** attempt, 20)
                print(f'  HTTP {response.status_code}; {wait}s 후 재시도')
                time.sleep(wait)
                continue

            response.raise_for_status()
            payload = response.json()

            found = {}
            for work in payload.get('results', []):
                doi = normalize_doi(work.get('doi'))
                if doi:
                    found[doi] = work
            return found

        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                wait = min(2 ** attempt, 20)
                print(f'  요청 오류: {exc}; {wait}s 후 재시도')
                time.sleep(wait)
            else:
                raise RuntimeError(f'OpenAlex API 요청 실패: {last_error}') from exc

    return {}


## 7. 입력 파일 점검 및 DOI 수집
이 셀을 실행하면 실제 조회 대상 DOI 수와 형식 오류 건수를 먼저 확인할 수 있습니다.


In [28]:
input_path = Path(INPUT_FILE)
output_path = Path(OUTPUT_FILE)

if not input_path.exists():
    raise FileNotFoundError(f'입력 파일이 없습니다: {input_path.resolve()}')

wb = load_workbook(input_path)

if SHEET_NAME not in wb.sheetnames:
    raise ValueError(f"시트 '{SHEET_NAME}'를 찾을 수 없습니다. 실제 시트: {wb.sheetnames}")

ws = wb[SHEET_NAME]
hmap = header_map(ws)

for required in (H_TITLE, H_AUTHOR, H_JOURNAL, H_DATE, H_DOI):
    if required not in hmap:
        raise ValueError(f'필수 열이 없습니다: {required}')

hmap = ensure_output_headers(ws, OUTPUT_HEADERS)
copy_header_style(ws, hmap[H_DOI], [hmap[h] for h in OUTPUT_HEADERS])

doi_rows = {}
invalid_doi_rows = []

for row in range(HEADER_ROW + 1, ws.max_row + 1):
    raw = ws.cell(row, hmap[H_DOI]).value
    if raw is None or not str(raw).strip():
        continue

    doi = normalize_doi(raw)
    if not doi:
        invalid_doi_rows.append(row)
        ws.cell(row, hmap['Cross-check']).value = 'INVALID DOI'
        continue

    doi_rows.setdefault(doi, []).append(row)

unique_dois = list(doi_rows)

print(f'전체 데이터 행: {ws.max_row - HEADER_ROW}')
print(f'유효 고유 DOI: {len(unique_dois)}')
print(f'형식 오류 DOI 행: {len(invalid_doi_rows)}')


전체 데이터 행: 721
유효 고유 DOI: 649
형식 오류 DOI 행: 0


## 8. OpenAlex 조회 실행


In [29]:
session = requests.Session()
session.headers.update({'User-Agent': 'OpenAlex-XLSX-Updater/1.0'})

all_results = {}
total_batches = (len(unique_dois) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(unique_dois), BATCH_SIZE):
    batch_no = i // BATCH_SIZE + 1
    batch = unique_dois[i:i + BATCH_SIZE]
    print(f'[{batch_no}/{total_batches}] OpenAlex {len(batch)}건 조회')
    result = query_batch(session, batch)
    all_results.update(result)
    print(f'  매칭 {len(result)}/{len(batch)}')
    time.sleep(REQUEST_DELAY)

print(f'\nOpenAlex 고유 DOI 매칭: {len(all_results)}/{len(unique_dois)}')


[1/7] OpenAlex 100건 조회
  매칭 100/100
[2/7] OpenAlex 100건 조회
  매칭 99/100
[3/7] OpenAlex 100건 조회
  매칭 100/100
[4/7] OpenAlex 100건 조회
  매칭 100/100
[5/7] OpenAlex 100건 조회
  매칭 99/100
[6/7] OpenAlex 100건 조회
  매칭 99/100
[7/7] OpenAlex 49건 조회
  매칭 48/49

OpenAlex 고유 DOI 매칭: 645/649


## 9. 결과 기록 및 `OpenAlex_Output.xlsx` 저장


In [30]:
matched_rows = 0
not_found_rows = 0
review_rows = 0

for doi, rows in doi_rows.items():
    work = all_results.get(doi)

    if work is None:
        for row in rows:
            ws.cell(row, hmap['Cross-check']).value = 'NOT FOUND'
        not_found_rows += len(rows)
        continue

    oa_title = work.get('display_name') or ''
    oa_author = get_first_author(work)
    oa_journal = get_journal(work)
    oa_date = work.get('publication_date')
    oa_doi = normalize_doi(work.get('doi'))

    for row in rows:
        ws.cell(row, hmap['OA 논문명']).value = oa_title
        ws.cell(row, hmap['OA 주저자']).value = oa_author
        ws.cell(row, hmap['OA 학술지명']).value = oa_journal
        ws.cell(row, hmap['OA 게재일']).value = oa_date
        ws.cell(row, hmap['Cited by']).value = work.get('cited_by_count')
        ws.cell(row, hmap['FWCI']).value = work.get('fwci')
        ws.cell(row, hmap['OpenAlex ID']).value = work.get('id')
        ws.cell(row, hmap['OA DOI']).value = oa_doi
        ws.cell(row, hmap['OpenAlex Updated']).value = work.get('updated_date')

        verdict = make_cross_check(
            ws.cell(row, hmap[H_TITLE]).value,
            ws.cell(row, hmap[H_AUTHOR]).value,
            ws.cell(row, hmap[H_JOURNAL]).value,
            ws.cell(row, hmap[H_DATE]).value,
            oa_title, oa_author, oa_journal, oa_date,
        )
        ws.cell(row, hmap['Cross-check']).value = verdict

        if verdict.startswith('CHECK'):
            review_rows += 1
        matched_rows += 1

widths = {
    'OA 논문명': 60,
    'OA 주저자': 24,
    'OA 학술지명': 42,
    'OA 게재일': 14,
    'Cited by': 12,
    'FWCI': 12,
    'OpenAlex ID': 28,
    'OA DOI': 38,
    'OpenAlex Updated': 18,
    'Cross-check': 36,
}

for name, width in widths.items():
    ws.column_dimensions[ws.cell(HEADER_ROW, hmap[name]).column_letter].width = width

for row in range(HEADER_ROW + 1, ws.max_row + 1):
    ws.cell(row, hmap['FWCI']).number_format = '0.00'
    ws.cell(row, hmap['Cited by']).number_format = '0'

wb.save(output_path)

print('==============================')
print('완료')
print('==============================')
print(f'OpenAlex 매칭 행 : {matched_rows}')
print(f'NOT FOUND       : {not_found_rows}')
print(f'CHECK 필요      : {review_rows}')
print(f'INVALID DOI     : {len(invalid_doi_rows)}')
print(f'출력 파일       : {output_path.resolve()}')


완료
OpenAlex 매칭 행 : 666
NOT FOUND       : 4
CHECK 필요      : 580
INVALID DOI     : 0
출력 파일       : D:\VSC\OpenAlex\OpenAlex_Output.xlsx


## 10. 선택 사항: CHECK/NOT FOUND 행 빠르게 확인


In [31]:
review_items = []
for row in range(HEADER_ROW + 1, ws.max_row + 1):
    status = ws.cell(row, hmap['Cross-check']).value
    if status and status != 'OK':
        review_items.append({
            'Excel row': row,
            'DOI': ws.cell(row, hmap[H_DOI]).value,
            '논문명': ws.cell(row, hmap[H_TITLE]).value,
            'OA 논문명': ws.cell(row, hmap['OA 논문명']).value,
            'Cross-check': status,
        })

print(f'검토 대상: {len(review_items)}건')
for item in review_items[:20]:
    print(item)


검토 대상: 584건
{'Excel row': 7, 'DOI': '10.1109/TASC.2005.849245', '논문명': 'Stability Study on Cryocooler- Cooled Superconducting Magnets', 'OA 논문명': 'Stability Study on Cryocooler-Cooled Superconducting Magnets', 'Cross-check': 'CHECK: AUTHOR'}
{'Excel row': 10, 'DOI': '10.13182/FST05-A929', '논문명': 'The Prediction of Tritium Level Reduction of Wolsong NPPs by Heavy Water Detritiation with WTRF', 'OA 논문명': 'The Prediction of Tritium Level Reduction of Wolsong NPPs by Heavy Water Detritiation with WTRF', 'Cross-check': 'CHECK: AUTHOR'}
{'Excel row': 12, 'DOI': '10.1016/j.fusengdes.2005.05.008', '논문명': 'Optimal configuration of neutron reflector in He-cooled molten Li blanket', 'OA 논문명': 'Optimal configuration of neutron reflector in He-cooled molten Li blanket', 'Cross-check': 'CHECK: AUTHOR'}
{'Excel row': 16, 'DOI': '10.1109/TASC.2005.869686', '논문명': 'Design and Electromagnetic Analysis of a Superconducting Diamagnetic Motor', 'OA 논문명': 'Design and Electromagnetic Analysis of a Supercondu